Total Number of CSV.GZ File

In [1]:
import os
import glob
import pandas as pd

# Path to your NebraskaLakes folder
input_dir = r"C:\\Users\\israt\\HD lab\\My Project\\Dataset\\Airsage Dataset\\UNL\\NebraskaLakes"

# Find all .csv.gz files recursively
all_gz_files = sorted(glob.glob(os.path.join(input_dir, "**", "*.csv.gz"), recursive=True))

# Count per folder (top-level under input_dir)
folder_counts = {}
for file in all_gz_files:
    # Relative path from input_dir
    rel_path = os.path.relpath(file, input_dir)
    # Folder name is first part of relative path
    folder = rel_path.split(os.sep)[0]
    folder_counts[folder] = folder_counts.get(folder, 0) + 1

# Convert to DataFrame for a nice table
summary_df = pd.DataFrame(sorted(folder_counts.items()), columns=["Folder", "Num_csv_gz"])

# Display results
print(summary_df.to_string(index=False))
print(f"\nTotal .csv.gz files found: {len(all_gz_files)}")


Folder  Num_csv_gz
202201         233
202202         233
202203         233
202204         233
202205         233
202206         233
202207         233
202208         233
202209         233
202210         233
202211         233
202212         233
202301         233
202302         233
202303         233
202304         233
202305         233
202306         233
202307         233
202308         233
202309         233
202310         233
202311         233
202312         233
202401         233
202402         233
202403         233
202404         233
202405         233
202406         233
202407         233
202408         233
202409         233
202410         233
202411         233
202412         233

Total .csv.gz files found: 8388


Accomodate Datafiles in One CSV

In [2]:
import os
import glob
import gzip
import pandas as pd
from io import StringIO

# Set input and output paths
input_dir = r"C:\\Users\\israt\\HD lab\\My Project\\Dataset\\Airsage Dataset\\UNL\\NebraskaLakes"
output_csv = r"C:\\Users\\israt\\HD lab\\My Project\\Dataset\\Airsage Dataset\\UNL\\NebraskaLakes_Cleaned.csv"
log_file = r"C:\\Users\\israt\\HD lab\\My Project\\Dataset\\Airsage Dataset\\UNL\\skipped_files_log.txt"

# Check if a file is actually gzip-compressed
def is_gzip(filepath):
    with open(filepath, 'rb') as f:
        return f.read(2) == b'\x1f\x8b'

# Read and parse structured content from file
def extract_structured_csv(file_path):
    open_func = gzip.open if is_gzip(file_path) else open
    with open_func(file_path, 'rt', encoding='utf-8') as f:
        lines = f.readlines()

    # Find the first non-comment line that looks like a header
    header_idx = next(i for i, line in enumerate(lines) if "," in line and not line.strip().startswith("#"))
    structured_data = "".join(lines[header_idx:])
    df = pd.read_csv(StringIO(structured_data))
    return df

# Discover all .csv.gz files
all_files = sorted(glob.glob(os.path.join(input_dir, "**", "*.csv.gz"), recursive=True))
print(f" Found {len(all_files)} files")

# Process each file
all_dfs = []
skipped = []

for file in all_files:
    try:
        df = extract_structured_csv(file)
        poi = os.path.basename(file).split(".")[0]
        if "poi" not in df.columns:
            df.insert(0, "poi", poi)
        all_dfs.append(df)
    except Exception as e:
        print(f" Skipped: {file} -- {e}")
        skipped.append((file, str(e)))

# Combine and save output
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df.to_csv(output_csv, index=False)
    print(f" Combined CSV saved to: {output_csv}")
else:
    print("No valid files to combine.")

# Save skipped log
if skipped:
    with open(log_file, "w", encoding="utf-8") as log:
        for path, reason in skipped:
            log.write(f"{path} -- {reason}\n")
    print(f" Skipped files log saved to: {log_file}")


🔍 Found 8388 files


C:\Users\israt\AppData\Local\Temp\ipykernel_32520\3123699562.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(all_dfs, ignore_index=True)


✅ Combined CSV saved to: C:\\Users\\israt\\HD lab\\My Project\\Dataset\\Airsage Dataset\\UNL\\NebraskaLakes_Cleaned.csv


Display All row from the generated CSV

In [5]:
!pip install tabulate


In [ ]:

import pandas as pd
from tabulate import tabulate  # pip install tabulate

df = pd.read_csv(
    r"C:\Users\israt\HD lab\My Project\Dataset\Airsage Dataset\UNL\NebraskaLakes_Cleaned.csv",
    dtype={3: str}  # column index 3 as string
)
#print(df.head(10))
print(tabulate(df.head(10), headers="keys", tablefmt="pretty", showindex=False))


+--------+-------------------------+----------+-------------+-------------+------------+----------+-------+----------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------+--------------+--------------+--------------------+--------------------+---------------+----------------+----------------+----------------+-----------------+----------+--------------------+--------------------+------------+------------+--------------------+------------+------------+--------------------+--------------------+--------------------+------------+------------+--------------------+------------+--------------------+------------+------------+------------+------------+------------+------------+--------------+------------+------------+--------------------+--------------+----

Roundup the floating value 

In [2]:
import pandas as pd
# Load the dataset again
file_path = r"C:/Users/israt/HD lab/My Project/Dataset/Airsage Dataset/UNL/NebraskaLakes_Cleaned.csv"
df = pd.read_csv(file_path, low_memory=False)

# Identify demographic columns (male_*, female_*)
demo_cols = [col for col in df.columns if col.startswith("male_") or col.startswith("female_")]

# Race/ethnicity groups
race_cols = [
    "white", "black", "native_american", "asian", "hawaiian_pacific",
    "other", "two_with_other", "two_ex_other_and_three_plus"
]

# Income groups
income_cols = [
    "income_0_10", "income_10_15", "income_15_20", "income_20_25", 
    "income_25_30", "income_30_35", "income_35_40", "income_40_45", 
    "income_45_50", "income_50_60", "income_60_75", "income_75_100", 
    "income_100_125", "income_125_150", "income_150_200", "income_over_200"
]

# Combine columns (only keep those actually in dataset)
all_round_cols = demo_cols + [col for col in race_cols if col in df.columns] + [col for col in income_cols if col in df.columns]

# Copy dataframe for presentation
df_int_all = df.copy()

# Round to nearest integer
df_int_all[all_round_cols] = df_int_all[all_round_cols].round(0).astype("Int64")

# Save the rounded version
output_path_int_all = r"C:/Users/israt/HD lab/My Project/Dataset/Airsage Dataset/UNL/NebraskaLakes_Cleaned_round.csv"
df_int_all.to_csv(output_path_int_all, index=False)

print("Saved rounded dataset to:", output_path_int_all)


Saved rounded dataset to: C:/Users/israt/HD lab/My Project/Dataset/Airsage Dataset/UNL/NebraskaLakes_Cleaned_round.csv


In [6]:
!pip install folium


   ---------------------------------------- 0.0/113.4 kB ? eta -:--:--
   --- ------------------------------------ 10.2/113.4 kB ? eta -:--:--
   ---------- ---------------------------- 30.7/113.4 kB 660.6 kB/s eta 0:00:01
   -------------------------------- ------- 92.2/113.4 kB 1.1 MB/s eta 0:00:01
   -------------------------------------- 113.4/113.4 kB 946.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/90.4 kB ? eta -:--:--
   ---------------------------------------- 90.4/90.4 kB 5.0 MB/s eta 0:00:00


View GeoJson File 

In [3]:
import geopandas as gpd
import folium
import tempfile, webbrowser

# 1) Load GeoJSON
gdf = gpd.read_file(r"C:/Users/israt/Downloads/polys2.geojson")
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# 2) Pick the name field
name_field = "WaterbodyName" if "WaterbodyName" in gdf.columns else "PublicWaterName"

# 3) Build the map
m = folium.Map(location=[41.5, -99.9], zoom_start=6, tiles="OpenStreetMap")

folium.GeoJson(
    gdf,
    tooltip=folium.GeoJsonTooltip(
        fields=[name_field],
        labels=False,     # <-- show ONLY the value, no "Name:" label
        sticky=True
    ),
    style_function=lambda x: {"color": "blue", "weight": 1, "fillOpacity": 0.5},
    highlight_function=lambda x: {"weight": 3, "color": "black"},
).add_to(m)

# 4) Open in browser and exit (no Ctrl+C needed)
with tempfile.NamedTemporaryFile("w", suffix=".html", delete=False, encoding="utf-8") as tmp:
    m.save(tmp.name)
    webbrowser.open_new_tab("file:///" + tmp.name.replace("\\", "/"))
